In [1]:
'''
Gán nhãn SRL tiếng Anh bằng AllenNLP.
Lưu kèm original_idx để tránh lệch index với VI sentences.

Input:
    data/translation/translation_pair.json

Output:
    data/silver/english_labels_2.json
'''

from allennlp.predictors.predictor import Predictor
import allennlp_models.tagging
import json
from tqdm import tqdm
import os


INPUT_FILE  = os.path.join("data", "translation", "translation_pair.json")
OUTPUT_FILE = os.path.join("data", "silver", "english_labels_2.json")

MODEL_URL = "https://storage.googleapis.com/allennlp-public-models/structured-prediction-srl-bert.2020.12.15.tar.gz"


def main():
    print("Gán nhãn SRL tiếng Anh")

    if not os.path.exists(INPUT_FILE):
        print(f"Không tìm thấy file tại {INPUT_FILE}")
        return

    try:
        predictor = Predictor.from_path(MODEL_URL)
    except Exception as e:
        print(f"Error khởi tạo model: {e}")
        return

    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        data = json.load(f)

    results = []
    skipped = 0

    for item in tqdm(data):
        # Lấy original_idx từ translation pair (được lưu ở bước translate)
        original_idx  = item.get("original_idx", None)
        sentence_en   = item.get("en", "")

        # Bỏ qua câu rỗng hoặc dịch thất bại — KHÔNG làm lệch index
        # vì ta dùng original_idx thay vì positional index
        if not sentence_en.strip() or sentence_en == "[TRANSLATION_FAILED]":
            skipped += 1
            continue

        try:
            output = predictor.predict(sentence=sentence_en)
            results.append({
                "original_idx": original_idx,   # <-- giữ lại để align đúng câu
                "sentence": sentence_en,
                "words": output["words"],
                "verbs": output["verbs"]
            })
        except Exception as e:
            print(f"Error tại câu idx={original_idx}: {e}")
            skipped += 1
            continue

    os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    print(f"Hoàn thành: {len(results)} câu được gán nhãn, {skipped} câu bị bỏ qua.")
    print(f"Kết quả lưu tại: {OUTPUT_FILE}")


if __name__ == "__main__":
    main()

D:\Downloads\Miniconda\envs\allennlp\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
D:\Downloads\Miniconda\envs\allennlp\lib\site-packages\google\api_core\_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.8.20). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
D:\Downloads\Miniconda\envs\allennlp\lib\site-packages\google\auth\__init__.py:52: FutureWarning: You are using a Python version 3.8 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3

Gán nhãn SRL tiếng Anh


error loading _jsonnet (this is expected on Windows), treating C:\Users\fah4i\AppData\Local\Temp\tmpcmv3bddm\config.json as plain json
Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertModel: ['cls.predictions.transform.dense.bias', 'cls.seq_relationship.weight', 'cls.predictions.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.decoder.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
100%|███████████████

Hoàn thành: 923 câu được gán nhãn, 0 câu bị bỏ qua.
Kết quả lưu tại: data\silver\english_labels_2.json
